# Auliya controlled B/C generalization V1 — Kaggle

Two independent native YOLO26s-Seg trainings (B and C), 300 training images per group, 75 evaluation images per group. Each fixed epoch-80 last.pt is evaluated on both B and C. All 21 classes, no group-A access. Attach only the existing extracted auliya-robusta-sni dataset; Internet and single GPU required. Run All trains both groups. Set GROUPS=["B"] or ["C"] for one run per session; Save Version and attach previous full Notebook Output for resume. G0 remains unchanged.


In [ ]:
# Settings
SCIENTIFIC_CODE_COMMIT = "4c992755484735acac0cdf511e6d89cec8cbf8bb"
AUTHORIZE_TRAINING = True
GROUPS = ["B", "C"]  # or just ["B"] / ["C"] in separate sessions
DEVICE = "0"
DEVELOPMENT_OVERRIDE = ""  # optional exact mounted development directory
METADATA_OVERRIDE = ""     # optional mounted official metadata ZIP
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from urllib.request import urlopen

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "auliya-controlled-generalization-v1"
REPO = WORK / "coffee-bean-detection-controlled-generalization"
DATA = WORK / "auliya-controlled-generalization-data"
DOWNLOAD = WORK / "auliya-controlled-generalization-assets"
assert INPUT.is_dir() and WORK.is_dir(), "Run on Kaggle"
assert GROUPS and len(set(GROUPS))==len(GROUPS) and set(GROUPS)<={"B","C"}
DOWNLOAD.mkdir(exist_ok=True)
DEV_SHA = "23d653ca2e98c185a76bb7b2095d6afc6401876cbe48fd219cb42bdc7ecbd360"
META_SHA = "798bb53b0cfad35dfcfbcc463fbcb79ff4af61f9807803d6bcc65d37f7c8feaa"
MODEL_SHA = "3da1d83e31caec96f9300eb4064f4f62882c133c7c264d63dfe61a7c197837a4"

def digest(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

if DEVELOPMENT_OVERRIDE:
    devs = [Path(DEVELOPMENT_OVERRIDE).resolve()]
else:
    devs = sorted(p.parent for p in INPUT.rglob("development_contract.json")
                  if p.parent.name == "development" and digest(p) == DEV_SHA)
if len(devs) != 1:
    raise RuntimeError(f"Expected exactly one verified development folder; found {devs}")
DEVELOPMENT = devs[0]
if not DEVELOPMENT.resolve().is_relative_to(INPUT.resolve()) or digest(DEVELOPMENT/"development_contract.json") != DEV_SHA:
    raise RuntimeError("Invalid mounted development contract")
print("Development:", DEVELOPMENT, flush=True)

prior = sorted(p for p in INPUT.rglob(PROJECT.name) if p.is_dir() and (p/"logs").is_dir())
if len(prior)>1:
    raise RuntimeError("Attach at most one previous Notebook Output for resume")
if prior and not PROJECT.exists():
    shutil.copytree(prior[0],PROJECT)
elif prior:
    print("Using current working output; not overwriting with prior")
PROJECT.mkdir(exist_ok=True)
(PROJECT/"logs").mkdir(exist_ok=True)

def fetch_verified(url, destination, expected):
    if destination.is_file() and digest(destination)==expected:
        return destination
    tmp = destination.with_suffix(".part")
    try:
        with urlopen(url,timeout=120) as source, tmp.open("wb") as sink:
            shutil.copyfileobj(source,sink)
        if digest(tmp)!=expected:
            raise RuntimeError(f"SHA mismatch: {destination}")
        tmp.replace(destination)
    finally:
        tmp.unlink(missing_ok=True)
    return destination

def run(cmd,log=None,cwd=None):
    cmd=list(map(str,cmd))
    print("$ "+" ".join(cmd),flush=True)
    if log is None:
        subprocess.run(cmd,check=True,cwd=cwd)
        return
    with Path(log).open("a",encoding="utf-8") as stream:
        with subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True)
                stream.write(line)
                stream.flush()
            if p.wait()!=0:
                raise RuntimeError(f"Command failed; see {log}")

if METADATA_OVERRIDE:
    METADATA=Path(METADATA_OVERRIDE).resolve()
    if not METADATA.is_relative_to(INPUT.resolve()) or digest(METADATA)!=META_SHA:
        raise RuntimeError("Official metadata override invalid")
else:
    METADATA=fetch_verified("https://zenodo.org/api/records/24346584/files/splits_metadata_results_scripts.zip/content",DOWNLOAD/"metadata.zip",META_SHA)
WEIGHTS=fetch_verified("https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt",DOWNLOAD/"yolo26s-seg.pt",MODEL_SHA)
print("Verified metadata and pretrained weights.")


In [ ]:
# Pinned scientific code, no package import from bundled Kaggle data.
if not REPO.exists():
    run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-detection.git",REPO])
if subprocess.run(["git","-C",str(REPO),"cat-file","-e",SCIENTIFIC_CODE_COMMIT+"^{commit}"],capture_output=True).returncode!=0:
    run(["git","-C",REPO,"fetch","--quiet","origin","codex/auliya-controlled-generalization"])
run(["git","-C",REPO,"checkout","--quiet","--detach",SCIENTIFIC_CODE_COMMIT])
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SCIENTIFIC_CODE_COMMIT
os.environ["PYTHONPATH"]=str(REPO/"src")+os.pathsep+os.environ.get("PYTHONPATH","")
os.environ["PYTHONUNBUFFERED"]="1"
os.environ["OMP_NUM_THREADS"]="2"
# Official fold intentionally uses newer Ultralytics than the pilot.
run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.210","scikit-learn","pytest"])
# Use subprocess to avoid stale imports from earlier Kaggle cells.
runtime=json.loads(subprocess.check_output([sys.executable,"-c",
    "import json,torch,ultralytics,numpy,cv2,sklearn,platform; print(json.dumps(dict(torch=str(torch.__version__),ultralytics=ultralytics.__version__,numpy=numpy.__version__,opencv=cv2.__version__,sklearn=sklearn.__version__,python=platform.python_version(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)))"],text=True).strip().splitlines()[-1])
assert runtime["ultralytics"]=="8.4.210",runtime
if AUTHORIZE_TRAINING and runtime["gpu"] is None:
    raise RuntimeError("Enable Kaggle GPU")
runtime["code_commit"]=SCIENTIFIC_CODE_COMMIT
marker=PROJECT/"environment.json"
if marker.exists() and json.loads(marker.read_text())!=runtime:
    raise RuntimeError("Runtime changed; do not resume in a different environment")
marker.write_text(json.dumps(runtime,indent=2))
print(runtime)
run([sys.executable,"-m","pip","freeze"],log=PROJECT/"logs/pip-freeze.txt")
run([sys.executable,"-m","pytest","-q","tests/test_auliya_official_fold.py","tests/test_auliya_representation_audit.py","tests/test_auliya_controlled_generalization.py"],cwd=REPO,log=PROJECT/"logs/tests.log")


In [ ]:
# Preflight: verify frozen splits against the original metadata and audited polygons.
RUNNER=[sys.executable,"-u","-m","coffee_detector.experiments.run_auliya_controlled_generalization",
        "--development",DEVELOPMENT,"--metadata",METADATA,"--data",DATA,
        "--weights",WEIGHTS,"--output",PROJECT/"runs","--device",DEVICE]
run(RUNNER+["--group","B"],cwd=REPO,log=PROJECT/"logs/preflight.log")
split=json.loads((PROJECT/"runs/split_contract.json").read_text())
assert split["test_A_images_accessed"] is False
assert {k:v["images"] for k,v in split["partitions"].items()}=={"B_train":300,"B_eval":75,"C_train":300,"C_eval":75}
print(json.dumps(split,indent=2))


In [ ]:
# Each source uses only its own internal eval during training.
if AUTHORIZE_TRAINING:
    for group in GROUPS:
        run(RUNNER+["--group",group,"--authorize-training"],cwd=REPO,log=PROJECT/"logs"/f"training_{group}.log")
else:
    print("Preflight only; set AUTHORIZE_TRAINING=True to train.")


In [ ]:
# Compare the two trained sources on exactly the same target eval list.
from zipfile import ZipFile, ZIP_DEFLATED
reports={}
for group in ("B","C"):
    p=PROJECT/"runs"/group/"validation_result.json"
    if p.exists():
        reports[group]=json.loads(p.read_text())
        assert reports[group]["test_A_images_accessed"] is False
        assert reports[group]["completed_epochs"]==80
        print("Source",group)
        for target,row in reports[group]["evaluations"].items():
            print("  Target",target,"Box18",row["box_map50_95_18"],"Hard5",row["hard5_box_ap50_95"],"Mask18",row["mask_map50_95_18"])
if len(reports)==2:
    keys=["box_map50_95_18","hard5_box_ap50_95","bottom3_box_ap50_95_18","worst_box_ap50_95_18",
          "mask_map50_95_18","hard5_mask_ap50_95","bottom3_mask_ap50_95_18","worst_mask_ap50_95_18"]
    contrasts={t:{k:reports[t]["evaluations"][t][k]-reports["C" if t=="B" else "B"]["evaluations"][t][k] for k in keys} for t in ("B","C")}
    summary={"complete":True,"internal_minus_cross_on_same_target":contrasts,"test_A_images_accessed":False}
else:
    summary={"complete":False,"available_sources":sorted(reports),"test_A_images_accessed":False}
(PROJECT/"comparison.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
archive=WORK/"auliya-controlled-generalization-report.zip"
with ZipFile(archive,"w",compression=ZIP_DEFLATED) as z:
    for p in sorted(PROJECT.rglob("*")):
        if p.is_file() and p.suffix.lower() not in {".pt",".part",".tmp"}:
            z.write(p,p.relative_to(PROJECT))
print("Upload report archive:",archive)
print("Save Version for full project and checkpoint pairs:",PROJECT)
